In [18]:
import pandas as pd
from bs4 import BeautifulSoup
import nltk
# nltk.download('punkt') - one time set up
# nltk.download('averaged_perceptron_tagger_eng') - one time set up
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import spacy
nlp = spacy.load("en_core_web_sm")

In [2]:
# save test file names sorted by volume
test_files = {"v1": [], "v2": [], "v3": [], "v4": []}
v1_docs = [1873, 1778, 1350, 1589, 1368, 2218, 2200, 1391, 1349, 1830]
v2_docs = [3379, 3255, 3195, 2966, 2884, 3423, 2943, 3280, 3042, 2653]
v3_docs = [3960, 3995, 3675, 3582, 4070, 3563, 4243, 3759, 4085, 4088]
v4_docs = [4751, 4624, 4407, 4538, 4828, 4444, 4702, 4896, 4576, 4483]
for i in v1_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v1"].append(fname)
for i in v2_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v2"].append(fname)
for i in v3_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v3"].append(fname)
for i in v4_docs:
    fname = "./test_files/article_" + str(i) + ".txt"
    test_files["v4"].append(fname)

In [3]:
# load in people table from scraping process
ppl_df = pd.read_csv('../ecg_interpreter/people_table.csv')
ppl_df = ppl_df.drop('Unnamed: 0', axis=1)
ppl_df.index.name = "ID"
ppl_df

,Designation,First Name,Alt First Name,Middle Name,Last Name,Alt Last Name,Title,Place,Organization
ID,,,,,,,,,
0,Individual,Willi,NaN,NaN,Aaron,NaN,NaN,NaN,NaN
1,Individual,Yitzhak,NaN,NaN,Aaron,NaN,NaN,NaN,NaN
2,Individual,Yehoshua,NaN,Moshe,Aaronson,NaN,NaN,NaN,NaN
3,Individual,Ervin,NaN,NaN,Abadi,Abady,NaN,NaN,NaN
4,Individual,Viktor,NaN,NaN,Abakumov,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
16840,Individual,Milovan,NaN,NaN,Žanić,NaN,NaN,NaN,NaN
16841,Individual,Antanas,NaN,NaN,Žilėnas,NaN,NaN,NaN,NaN
16842,Individual,Vladas,NaN,NaN,Žilėnas,NaN,NaN,NaN,NaN


Name Combinations:
- all reg
    - fml - first, middle, last
    - fl - first, last (regardless of whether middle is known or not)
- alt last
    - fmal - first, middle, alt last
    - fal - first, alt last (regardless of whether middle is known or not)
- alt first and last
    - afmal - alt first, middle, alt last
    - afal - alt first, alt last (regardless of whether middle is known or not)
- alt first
    - afml - alt first, middle, last
    - afl - alt first, last (regardless of whether middle is known or not)

In [4]:
# dict to store instances where there are multiple people that fit the criteria
# key: sentence, value: df of name options
mult_ppl_options = {}

In [5]:
# helper function - if first name found, determine if next words are middle or last name
# checks for fmal, afmal, fal, or afal
def is_full_name_altlname(filtered_df, words_lst, fname_i, sent):
    next_w = words_lst[fname_i + 1]
    # check if next word is a corresponding middle name - trying to find fmal or afmal
    found_mnames = filtered_df[filtered_df['Middle Name'] == next_w]
    if len(found_mnames) != 0:
        # check if word after mname is a corresponding last name - trying to find fmal or afmal
        next2_w = words_lst[fname_i + 2]
        found_lnames = found_mnames[found_mnames['Alt Last Name'] == next2_w]
    # check if next word is a corresponding last name - trying to find fal or afal
    else:
        found_lnames = filtered_df[filtered_df['Alt Last Name'] == next_w]
        
    # found one row with valid fml or fl name
    if len(found_lnames) == 1:
        found_id = found_lnames.index[0]
        print("FINAL ID", found_id)
        return found_id
    # did not find valid name, classify as NOT PERSON NAME (-1)
    elif len(found_lnames) == 0:
        print("NOT PERSON NAME")
        return -1
    # found multiple options, parse further (later change to call other helper function)
    else:
        print("TOO MANY PEOPLE")
        mult_ppl_options[sent] = filtered_df
        return -2

In [6]:
# helper function - if first name found, determine if next words are middle or last name
# checks for fml, afml, fl, or afl
def is_full_name(filtered_df, words_lst, fname_i, sent):
    next_w = words_lst[fname_i + 1]
    # check if next word is a corresponding middle name - trying to find fml or afml
    found_mnames = filtered_df[filtered_df['Middle Name'] == next_w]
    if len(found_mnames) != 0:
        # check if word after mname is a corresponding last name - trying to find fml or afml
        next2_w = words_lst[fname_i + 2]
        found_lnames = found_mnames[found_mnames['Last Name'] == next2_w]
    # check if next word is a corresponding last name - trying to find fl or afl
    else:
        found_lnames = filtered_df[filtered_df['Last Name'] == next_w]
        
    # found one row with valid fml or fl name
    if len(found_lnames) == 1:
        found_id = found_lnames.index[0]
        print("FINAL ID", found_id)
        return found_id
    # did not find valid name with reg lname, check alt lname
    elif len(found_lnames) == 0:
        print("CHECK ALT LAST NAME")
        return is_full_name_altlname(filtered_df, words_lst, fname_i, sent)
    # found multiple options, parse further (later change to call other helper function)
    else:
        print("TOO MANY PEOPLE")
        mult_ppl_options[sent] = filtered_df
        return -2

In [7]:
# helper function - determine if given sent contains a person name
def is_key(sent):
    local_key_sents = {}
    result = -1
    print(sent)
    words = word_tokenize(sent)
    for i, w in enumerate(words):
        found_fnames = ppl_df[ppl_df['First Name'] == w]
        if len(found_fnames) != 0:
            result = is_full_name(found_fnames, words, i, sent)
            print("REG RESULT:", result)
#             print(found_fnames)
        # didn't find the correct person from fname, check afname
        if result == -1:
            found_afnames = ppl_df[ppl_df['Alt First Name'] == w]
            if len(found_afnames) != 0:
                result = is_full_name(found_afnames, words, i, sent)
                print("ALT RESULT:", result)
        # didn't find correct person from fname or afname, check if lname
        if result == -1:
            found_lnames = ppl_df[ppl_df['Last Name'] == w]
            if len(found_lnames) != 0:
                mult_ppl_options[sent] = found_lnames
                result = -2
        if result != -1 and result != -2:
            if result not in local_key_sents:
                local_key_sents[result] = []
            if sent not in local_key_sents[result]:
                local_key_sents[result].append(sent)
    return local_key_sents

In [8]:
# helper function - split body into sentences, check if key, save results in dict
def find_key_sents(body):
    # key: person ID, value: list of sentences containing name
    key_sents = {}
    # split body into sentences
    sents = sent_tokenize(body)
    for s in sents:
        key_sents |= is_key(s)
    return key_sents

In [9]:
# helper function - write key sentences to file
def save_sentences(title, key_dct):
    with open("tagged_people_sentences.txt", "a") as f:
        f.write(f"ARTICLE: {title}\n")
        for key, item in key_dct.items():
            f.write(f"NAME: {ppl_df.iloc[key].to_string()}\n")
            for sent in item:
                f.write(f"SENTENCE: {sent}\n")
                f.write(f"{nltk.pos_tag(word_tokenize(sent))} \n\n")
#         f.write("UNDETERMINED SENTENCES:\n")
#         for key, item in mult_ppl_options.items():
#             f.write(f"SENTENCE: {key}\n\n")
#             f.write(item.to_string())
#             f.write("\n\n")
        f.write("------------------------------------\n\n")

# helper function - write volume number to file
def new_volume(vol):
    with open("tagged_people_sentences.txt", "a") as f:
        f.write("----------------------------------------------------------------------\n")
        f.write(f"VOLUME: {vol}\n\n")

In [10]:
all_key_sents = {}
for vol, articles in test_files.items():
    new_volume(vol)
    for article_path in articles:
        mult_ppl_options = {}
        # save article title and text
        with open(article_path, "r", encoding="utf-8") as f:
            title = f.readline().replace("ARTICLE: ", "")
            text = f.read()
        # get sentences with people names - print out & save to file
        article_key_sentences = find_key_sents(text)
        print(article_key_sentences)
        all_key_sents |= article_key_sentences
        save_sentences(title, article_key_sentences)


An Organisation Todt (OT) police custody camp (Polizeihaftlager) subcamp of the Hinzert main concentration camp was located in Uthlede near Wesermünde in the Prussian province of Hannover.
The camp files refer to the camp for the first time on April 26, 1940.
The police prisoners held here were workers of the OT Senior Construction Administration (Oberbauleitung, OBL) in Bremen.
They were interned for a variety of disciplinary offenses—mainly work absenteeism, thefts from “comrades,” fights, assaults on superiors, statements against National Socialism, and the like.
They were interned for a maximum of three months.
Prisoners with longer sentences were held at Hinzert.
Police custody camps came into being at the end of 1939 in order to deal with the growing disciplinary problems during the construction of the West Wall and the Reichsautobahn (RAB).
“The common aim in establishing such camps was to re-educate the mostly young, conscripted OT workers into a National Socialist way of life

They reached the Zwodau camp on March 6, 1945; 19 women died en route, and more died at the camp.
Part of the transport reached the Flossenbürg concentration camp subcamp   Helmbrechts at the same time.
Jeschke turned the prisoners over to the camp command.
Locked in unfinished barracks, without medical care, and miserably nourished, masses of them became sick.
They were not even assigned to any work.
They lived under these conditions for six weeks.
The camp authorities decided to continue the evacuation because the U.S. Army was approaching.
On April 13, an evacuation column of 581 women set out toward Zwodau, where it was joined by another group of prisoners.
In all, the transport that set out from Zwodau numbered 700 prisoners.
They reached Wallern (Volary) on May 4, 1945.
Approximately 300 women remained.
They were locked in a shed.
The local people were forbidden to help them at all.
Some of the women were unable to march by this point, so the SS men demanded that the mayor provid

Since the concrete would thicken within a week, it   allowed the gravel to be removed by sending trains into the tunnels.
By opening flaps in the tunnel roof, the railway wagons would be filled with the gravel.
This system had the advantage that the gravel could be used again for concrete pours or for building another gravel wall.
The bunker could be extended by single segments as required.
Once the gravel was completely removed from the concrete, completion of the interior could immediately commence.
The biggest problem was the lack of labor.
A large number of the forced laborers made available for the construction of the bunker were Hungarian Jews.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
From July 1944, there arose in the nearby vicinity of Mühldorf am Inn four camps subordinate to the Dachau concentration camp, two larger camps for about 2,000 to 3,000 prisoners at Mettenheim near Mühldorf and a forest camp at Ampfing, as well as two smaller camps, one at Mittergars and t

The authorities occupied the grounds of an abandoned factory, Firma August Wellner & Söhne, without paying rent.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
Although Zschorlau had a capacity for holding approximately 500 detainees, the total population did not exceed 207.
Eighty percent of the prisoners were Communists.
One prisoner was Jewish.
The commandant, SS-Scharführer Robert Philipp Weissmann, was a fanatical National Socialist.
FINAL ID 15868
REG RESULT: 15868
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
The camp administration used Communist Party funds for the procurement of prisoners’ eating utensils, clothing, and other supplies.
Confined to an industrial waste pit, the prisoners were chained by the legs to prevent escape.
Zschorlau’s harsh conditions and rough interrogations caused the deaths of Otto Hempel, Paul Höhl, Albert Höhnel, Erich Pilz, and Alfred Schädlich.
FINAL ID 5640
REG RESULT: 5640
FINAL ID 6073
REG RESULT: 6073
FINAL ID 6076
REG RESULT: 6076
F

From then on, the women were used for heavy and dangerous physical work outside the camp, including cleanup work in Berlin and digging antitank ditches to defend the city.
Rations sank dramatically once production at HFW ceased in January 1945.
The hygienic conditions also dramatically deteriorated, as the water supply system broke down.
“Many young women died very quickly of tuberculosis.
Usually the ill, and those whose performance at work was insufficient, were sent back to the camp and I subsequently learned of their deaths,”13 Cartier-Worms reported.
There was a sick ward in Assembly Hall 7, attended to by two women prisoners, but it only dealt with minor illnesses and injuries.
The seriously ill were taken at first to Ravensbrück and, beginning in 1945, to the women’s subcamp of Sachsenhausen called Oranienburg (Auer-Werke).
There were probably multiple deaths on the grounds of the Schönefeld subcamp, and a former female guard remembers at least one.
The SS change of status repor

...

CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
Since there was no information about the accused available in Ludwigsburg, the matter was passed on to the Central Office of the State of Nordrhein-Westfalen for the Processing of Criminal Proceedings for National Socialist Mass Crimes at the Public Prosecutor’s Office in Dortmund.
It opened an investigation of a captain and a sergeant in the camp for murder or aiding and abetting murder, because they had built neither air-raid shelters nor ditches for the prisoners.
In addition, they had forbidden the prisoners to leave the barracks during the air raids, causing the deaths of about 245 men.
The investigation was closed in November 1992 because the perpetrators could not be found.
{np.int64(15388): ['They were Oberstleutnant Kurt Vilmar (b.'], np.int64(3033): ['1882), from September 29, 1939, to April 14, 1940; Oberst Wilhelm Eck (b.'], np.int64(12301): ['1870), from April 15, 1940, to June 21, 1942; Oberstleutnant Otto Rotenberg (b.']

Departures due to repatriations were repeatedly offset by arrivals from other camps.
Oflag XVII A at Allentsteig.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
General view of the camp, August 1940.COURTESY OF ICRC.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
Oflag XVII A’s staff consisted of 140 men in October 1943, plus 382 men of Company 2 of Reserve Battalion (Landesschützenbataillon) 893 as the guard battalion.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
The camp commandants were the distinctly pro-French old Austrian Generalmajor z.V.
Anton Unger and—in the wake of the “Great Escape”—from late 1943–early 1944, Generalmajor Hans Ehrenberg, a native of Saxony.
FINAL ID 15181
REG RESULT: 15181
FINAL ID 3085
REG RESULT: 3085
The mortality rate among the French officers was not high, although it began to increase toward the end of the war.
The total number of French prisoners who met their death in the camp environment (including the military hospitals in Allentstei

The Germans recaptured all but 10–13 of the escapees, and proceeded to punish all the prisoners in the camp, collectively.
The Germans placed the recaptured men under close arrest for 10–21 days, while civilians who had helped with the escape were severely punished (18–30 months in prison).
Ehrenberg, the new German camp commandant, created an additional “ring of defensive fortification” in April 1944, to protect against possible relief attacks or insurrections in the camp.
Some French officers suffered and died because of their Jewish background.
Lieutenant Lantz perished in Auschwitz in March 1942, having been deported there after his repatriation or demobilization.
A similar fate befell at least seven other French officers: Lieutenants Marx (Auschwitz), Cassagne (April 1945 in Neuengamme), Goudchaux (September 1944), Leborgne (during deportation), de Liocourt (May 1944 in Buchenwald), Henri Jonval (1945 during deportation), and Capitaine Dumesne (died returning from deportation).
CH

In the second half of 1943, the two sections of Frontstalag 221 in the region of Bordeaux and in Rennes together held almost 11,000 prisoners, making this Frontstalag by far the largest in France, with nearly a third of all remaining French colonial prisoners.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
In Saint-Médard, Frontstalag 221 included the Camp des As along the Jalles River.
Other subcamps were the nearby Camp des Annamites, called after Indochinese civilian workers who had worked nearby, and various other sites around Saint-Médard and neighboring towns, such as the Camp de Souge to the west and the Camp de Germignan in Le-Taillan-Médoc, to the northeast.
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
CHECK ALT LAST NAME
NOT PERSON NAME
REG RESULT: -1
Little documentation exists for the first months of Frontstalag 221’s existence.
Like most other camps during this early period, it appears to have been overpopulated and undersupplied.
The French army physician Dr. Ho

Hollecker pointed out that the Germans were very generous in dismissing prisoners even though many diseases were feigned; he knew of prisoners rubbing sand, tobacco leaves, or quinine salt into their eyes or skin so that it looked as if they were developing a trachoma or a rash indicating a dangerous infectious disease.
Hollecker also described the establishment of a special section in the hospital staffed by 12 German physicians who specialized in tropical diseases.
Prisoners from other Frontstalags were often sent to this hospital for treatment.
The French physicians worked together with the Germans.
Soon after Hollecker’s dismissal from captivity, however, the Germans separated one building of the hospital from the Frontstalag, closed it to the French physicians, and used it as a secret training and research facility.
This hospital attracted Nazi researchers interested in demonstrating racial differences through physical and pathological characteristics, such as Dr. Wolfgang Abel, a

In [61]:
def get_action(name, sents):
    for s in sents:
        # tokenize s using spacy
        token_s = nlp(s)
        # get root verb
        for token in token_s:
            # check if root: token.dep_ == "ROOT"
            if token.pos_ == "VERB":
                action = token.text
                print(action)
                action_lemma = token.lemma_
#                 print(action, action_lemma, token.dep_)
                for child in token.children:
                    print(child)
                    if (child.dep_ == "nsubj" or child.dep_ == "nsubjpass") and child.ent_type_ == "PERSON":
                        print(child.text, token.lemma_)
                        
#         ws = word_tokenize(s)
#         tagged = nltk.pos_tag(ws)
#         vbz = [w for w, pos in tagged if pos == "VBZ"]
#         print(vbz)
# LOOK INTO spacy's DependencyMatcher

In [62]:
for pid, sents in all_key_sents.items():
    # format name
    person = ppl_df.iloc[pid]
    name = ((person["First Name"] + " ") if pd.notna(person["First Name"]) else "")
    name += (("(" + person["Alt First Name"] + ") ") if pd.notna(person["Alt First Name"]) else "")
    name += ((person["Middle Name"] + " ") if pd.notna(person["Middle Name"]) else "")
    name += ((person["Last Name"] + " ") if pd.notna(person["Last Name"]) else "")
    name += (("(" + person["Alt Last Name"] + ") ") if pd.notna(person["Alt Last Name"]) else "")
    print(name)
    get_action(name, sents)
    break

Anna Charzykow 
testifies
One
,
taken
taken
that
on
camp
was
over
had
testifies
,
women
pass
,
made
.
pass
to
naked
before
made
while
men
notes


In [12]:
all_key_sents

{np.int64(2108): ['One of the prisoners, Anna Charzykow, testifies that on the day the camp was taken over, all the women had to pass totally naked before each SS man in the general hall, while the SS men made notes.'],
 np.int64(6432): ['Anna Jon held the position of Lagerführerin (camp leader).'],
 np.int64(15363): ['The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre, Hildegard Kuehn, Helga Siebert, and Anna Hempel.'],
 12917: ['The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre, Hildegard Kuehn, Helga Siebert, and Anna Hempel.'],
 np.int64(7963): ['The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre, Hildegard Kuehn, Helga Siebert, and Anna Hempel.'],
 np.int64(13659): ['The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre, Hildegard Kuehn, Helga Siebert, and Anna Hempel.'],
 np.int64(5637): ['The staff mentioned by former prisoners included Anna Viebig, Waltrand Schirmre